### **Import Library**

In [25]:
import pandas as pd
import numpy as np

### **Setting**

In [26]:
# Prevent wrapping rows onto new lines
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

# Format scientific notation into standard numbers with commas
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

### **Read File**

In [27]:
annual_gross_value = pd.read_csv("../Data/Annual Gross Value Added.csv")
employment = pd.read_csv("../Data/Gross Value Added and Employment.csv")
tax_and_subsidies = pd.read_csv("../Data/Gross Value Added, Taxes and Subsidies.csv")

### **List out their features**

In [28]:
# Display column names and data types
print("--- Annual Gross Value Added ---")
print(annual_gross_value.dtypes)

print("\n--- Gross Value Added and Employment ---")
print(employment.dtypes)

print("\n--- Gross Value Added, Taxes and Subsidies ---")
print(tax_and_subsidies.dtypes)

--- Annual Gross Value Added ---
Statistic Label          str
Year                   int64
Economic Activity        str
UNIT                     str
VALUE                float64
dtype: object

--- Gross Value Added and Employment ---
Statistic Label           str
Year                    int64
NACE Rev. 2 Sector        str
NUTS 2 Region             str
UNIT                      str
VALUE                 float64
dtype: object

--- Gross Value Added, Taxes and Subsidies ---
Statistic Label        str
Year                 int64
NUTS 2 Region          str
UNIT                   str
VALUE              float64
dtype: object


### **Try to find out the common between three db**

In [29]:
# To inspect the exact unique entries from your loaded DataFrames
for name, df in [("Annual GVA", annual_gross_value), 
                 ("Employment", employment), 
                 ("Taxes & Subsidies", tax_and_subsidies)]:
    print(f"=== {name} ===")
    print("Statistic Labels:", df["Statistic Label"].unique().tolist())
    print("Years:", sorted(df["Year"].unique().tolist()))
    print("Units:", df["UNIT"].unique().tolist())
    print("\n")

=== Annual GVA ===
Statistic Labels: ['Gross Value Added at Current Basic Prices', 'Gross Value Added at Constant Basic Prices']
Years: [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Units: ['Euro Million']


=== Employment ===
Statistic Labels: ['Gross Value Added (GVA) at Basic Prices', 'Total Persons at Work', 'Employees at Work', 'Self-Employed at Work', 'Hours Worked']
Years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Units: ['Euro Million', 'Number', 'Thousand']


=== Taxes & Subsidies ===
Statistic Labels: ['Compensation of Employees', 'Gross Value Added (GVA) at Factor Cost', 'Non-Product Taxes', 'Non-Product Subsidies', 'Gross Value Added (GVA) at Basic Prices', 'Gross Value Added (GVA) at Basic Prices per Person', 'Product Taxes', '

#### We can notice the common year range for three db is from 2000 to 2024

### **Cleaning Dataframe**

In [30]:
# We just need data from 2000 to 2024
annual_gross_value = annual_gross_value[
    (annual_gross_value['Year'] >= 2000) & (annual_gross_value['Year'] <= 2024)
]

### Merge the feature of unit and value

In [31]:
# Define unit multipliers
unit_multipliers = {
    'Euro Million': 1_000_000,
    'Persons Thousand': 1_000,
    'Thousand': 1_000,
    'Euro': 1,
    'Number': 1
}

# Function to scale value and remove unit col
def normalize_value_and_drop_unit(df):
    # Map the unit to its numeric multiplier
    multipliers = df['UNIT'].map(unit_multipliers)
    
    df['VALUE'] = df['VALUE'] * multipliers
    
    # Drop the unit col
    df = df.drop(columns=['UNIT'])
    
    return df

# Apply to df
annual_gross_value = normalize_value_and_drop_unit(annual_gross_value)
employment = normalize_value_and_drop_unit(employment)
tax_and_subsidies = normalize_value_and_drop_unit(tax_and_subsidies)

print("=== Annual Gross Value Added ===")
print(annual_gross_value.head())

print("\n=== Gross Value Added and Employment ===")
print(employment.head())

print("\n=== Gross Value Added, Taxes and Subsidies ===")
print(tax_and_subsidies.head())

=== Annual Gross Value Added ===
                               Statistic Label  Year                                  Economic Activity         VALUE
430  Gross Value Added at Current Basic Prices  2000              Agriculture, forestry and fishing (A) 2,726,000,000
431  Gross Value Added at Current Basic Prices  2000  Crop and animal production, hunting and relate... 2,472,000,000
432  Gross Value Added at Current Basic Prices  2000                          Forestry and logging (02)    82,000,000
433  Gross Value Added at Current Basic Prices  2000                       Fishing and aquaculture (03)   172,000,000
434  Gross Value Added at Current Basic Prices  2000                           Mining and quarrying (B)   479,000,000

=== Gross Value Added and Employment ===
                           Statistic Label  Year                 NACE Rev. 2 Sector         NUTS 2 Region         VALUE
0  Gross Value Added (GVA) at Basic Prices  2000  Agriculture, Forestry and Fishing  Northern and

### Check Missing Value

In [32]:
print("=== Annual Gross Value Added ===")
print(annual_gross_value.isnull().sum())

print("\n=== Gross Value Added and Employment ===")
print(employment.isnull().sum())

print("\n=== Gross Value Added, Taxes and Subsidies ===")
print(tax_and_subsidies.isnull().sum())

=== Annual Gross Value Added ===
Statistic Label        0
Year                   0
Economic Activity      0
VALUE                280
dtype: int64

=== Gross Value Added and Employment ===
Statistic Label        0
Year                   0
NACE Rev. 2 Sector     0
NUTS 2 Region          0
VALUE                 48
dtype: int64

=== Gross Value Added, Taxes and Subsidies ===
Statistic Label    0
Year               0
NUTS 2 Region      0
VALUE              4
dtype: int64


### Remove Missing Value

In [34]:
# Remove Missing Values
annual_gross_value = annual_gross_value.dropna(subset=['VALUE']).reset_index(drop=True)
employment = employment.dropna(subset=['VALUE']).reset_index(drop=True)
tax_and_subsidies = tax_and_subsidies.dropna(subset=['VALUE']).reset_index(drop=True)

print("=== Annual Gross Value Added ===")
print(annual_gross_value.isnull().sum())

print("\n=== Gross Value Added and Employment ===")
print(employment.isnull().sum())

print("\n=== Gross Value Added, Taxes and Subsidies ===")
print(tax_and_subsidies.isnull().sum())

=== Annual Gross Value Added ===
Statistic Label      0
Year                 0
Economic Activity    0
VALUE                0
dtype: int64

=== Gross Value Added and Employment ===
Statistic Label       0
Year                  0
NACE Rev. 2 Sector    0
NUTS 2 Region         0
VALUE                 0
dtype: int64

=== Gross Value Added, Taxes and Subsidies ===
Statistic Label    0
Year               0
NUTS 2 Region      0
VALUE              0
dtype: int64
